In [1]:

import yfinance as yf
data=yf.download("RELIANCE.NS", period="5y", multi_level_index=False)
print(data)

[*********************100%***********************]  1 of 1 completed

                  Close         High          Low         Open    Volume
Date                                                                    
2021-10-04  1157.726074  1166.195597  1149.075293  1156.299372  10918552
2021-10-05  1181.753418  1183.021624  1153.740388  1157.250538  13533634
2021-10-06  1159.401978  1186.622290  1155.438983  1186.622290  10258881
2021-10-07  1165.086060  1177.020448  1162.957328  1175.321919   6336665
2021-10-08  1209.857056  1215.812929  1169.207606  1170.906015  15772493
...                 ...          ...          ...          ...       ...
2026-09-25  1226.000000  1227.400024  1210.500000  1210.500000  13138735
2026-09-28  1197.599976  1219.699951  1197.000000  1218.000000  13602349
2026-09-29  1182.000000  1198.300049  1181.800049  1193.800049  24191416
2026-09-30  1187.000000  1196.500000  1181.699951  1182.000000  16376789
2026-10-01  1167.699951  1183.900024  1160.800049  1180.099976  16771221

[1240 rows x 5 columns]


In [2]:
def get_sma(closing_price,window):
    sma=   closing_price.rolling(window=window).mean()
    return sma
sma=get_sma(data['Close'],window=9)
sma

Date
2021-10-04            NaN
2021-10-05            NaN
2021-10-06            NaN
2021-10-07            NaN
2021-10-08            NaN
                 ...     
2026-09-25    1236.288900
2026-09-28    1232.100003
2026-09-29    1225.655558
2026-09-30    1219.333333
2026-10-01    1212.811103
Name: Close, Length: 1240, dtype: float64

In [3]:
import pandas_ta_classic as ta

def get_ema(closing_price,window):
    ema=ta.ema(closing_price,window)
    return ema
ema=get_ema(data['Close'],window=9)
ema

Date
2021-10-04            NaN
2021-10-05            NaN
2021-10-06            NaN
2021-10-07            NaN
2021-10-08            NaN
                 ...     
2026-09-25    1240.205380
2026-09-28    1231.684300
2026-09-29    1221.747440
2026-09-30    1214.797952
2026-10-01    1205.378352
Name: EMA_9, Length: 1240, dtype: float64

In [4]:
from backtesting import Backtest, Strategy

class sma_strategy(Strategy):
    n1=50
    n2=20

    def init(self):
        self.sma1=self.I(get_sma,self.data.Close.s,self.n1)
        self.ema1=self.I(get_ema,self.data.Close.s,self.n2)

    def next(self):
        current_closing_price=self.data.Close[-1]
        previous_closing_price=self.data.Close[-2]
        current_sma=self.sma1[-1]
        previous_sma=self.sma1[-2]
        current_ema=self.ema1[-1]
        previous_ema=self.ema1[-2]

        if current_ema>current_sma and previous_sma>previous_ema:
            self.position.close()
            self.buy()
        elif current_ema<current_sma and previous_sma<previous_ema:
            self.position.close()
            self.sell()

b1=Backtest(data,sma_strategy,cash=10000)
result=b1.run()
b1.plot()

/Users/algo trading 2026/batch 38/.venv/lib/python3.14/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

/var/folders/gf/2xw81y_n41x8mbjy1d6lp0540000gn/T/ipykernel_45449/3788453634.py:27: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  result=b1.run()


GridPlot(id='p1320', ...)

In [5]:
result

Start                     2021-10-04 00:00:00
End                       2026-10-01 00:00:00
Duration                   1823 days 00:00:00
Exposure Time [%]                    93.14516
Equity Final [$]                    5082.6768
Equity Peak [$]                   10062.50175
Return [%]                          -49.17323
Buy & Hold Return [%]                 8.63691
Return (Ann.) [%]                   -12.85907
Volatility (Ann.) [%]                16.52497
CAGR [%]                               -12.68
Sharpe Ratio                         -0.77816
Sortino Ratio                        -0.95499
Calmar Ratio                         -0.24615
Alpha [%]                           -48.33379
Beta                                 -0.09719
Max. Drawdown [%]                   -52.24074
Avg. Drawdown [%]                   -52.24074
Max. Drawdown Duration     1718 days 00:00:00
Avg. Drawdown Duration     1718 days 00:00:00
# Trades                                   31
Win Rate [%]                      